# ColdSite-DTI — DAVIS training grid (Colab, T4)

Trains the audit's 36 cells: **3 models x 4 splits x 3 seeds**, DAVIS, binary task.

| | |
|---|---|
| Runtime needed | **T4 GPU** (Runtime -> Change runtime type -> T4 GPU) |
| Total compute | ~12-30 GPU-hours |
| Sessions | expect 4-8, because Colab free disconnects |
| Resumable | yes - finished cells are skipped, so just re-run this notebook |

**A TPU runtime will not work.** `torch.cuda.is_available()` is `False` there, so every
trainer silently falls back to Colab's 2 vCPUs and the grid takes ~1,000 hours instead
of ~20. Cell 1 checks for this and stops.

Everything writes to Google Drive, so a disconnect costs you one cell, not one session.

## 1. Check the runtime

Stops immediately if this is not a GPU.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No CUDA device. Runtime -> Change runtime type -> T4 GPU. '
    'A TPU runtime falls back to CPU and the grid takes ~1000 hours.'
)

name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU     : {name}')
print(f'memory  : {total_gb:.1f} GB')
print(f'torch   : {torch.__version__}')

# Measured peak activation memory, forward + backward, 1000-residue protein:
#   ColdSite-DTI  batch 64 -> 8.7 GB      HyperAttentionDTI batch 32 -> ~5 GB
#   DeepDTA       batch 256 -> 0.7 GB
if total_gb < 14:
    print('\nWARNING: under 14 GB. Use the small-batch settings in the training cells:')
    print('  COLDSITE_BATCH=16 HAT_BATCH=8 HAT_ACCUM=4')

## 2. Mount Drive

Checkpoints and results live here so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Unchanged from earlier runs on purpose: this is where your existing
# DeepDTA checkpoints already live. Changing it would make them look
# missing even though nothing was lost.
DRIVE_RESULTS = '/content/drive/MyDrive/coldsite-results'
# New: the two other things a Colab runtime reset silently wipes, because
# neither lives in git (data/splits/* and the DeepDTA source files are
# gitignored) or in Drive. A reset used to mean re-downloading and
# re-building both from scratch before training could resume.
DRIVE_SPLITS = '/content/drive/MyDrive/coldsite-splits'
DRIVE_DEEPDTA_DATA = '/content/drive/MyDrive/coldsite-deepdta-data'

for path in (DRIVE_RESULTS, DRIVE_SPLITS, DRIVE_DEEPDTA_DATA):
    os.makedirs(path, exist_ok=True)

print('results     ->', DRIVE_RESULTS)
print('splits      ->', DRIVE_SPLITS)
print('deepdta src ->', DRIVE_DEEPDTA_DATA)
print('existing checkpoints:',
      len([f for f in os.listdir(DRIVE_RESULTS) if f.endswith('.pt')]))

## 3. Clone the repo and install

Only `tabulate` and `subword-nmt` are missing from Colab's image. Torch, numpy, pandas,
scikit-learn, scipy and matplotlib are already there, and reinstalling torch risks
breaking the CUDA build.

**Results, splits and the DeepDTA source files all live in Drive**, symlinked into the
clone. A Colab free-tier disconnect wipes everything under `/content` - a fresh clone
on reconnect used to mean re-downloading the DeepDTA files and rebuilding the splits
before training could resume. Now it costs nothing: this cell's `ensure_symlink` re-links
to the same Drive folders, and cells 5-6 below skip anything already present.

In [ ]:
BRANCH = 'claude/repo-analysis-direction-2hiaxx'
REPO = 'https://github.com/udayraj1238/ColdSite-DTI.git'

%cd /content
if not os.path.exists('/content/ColdSite-DTI'):
    !git clone --branch {BRANCH} {REPO}
%cd /content/ColdSite-DTI
!git checkout {BRANCH} && git pull origin {BRANCH}

!pip install -q tabulate subword-nmt


def ensure_symlink(local_path, drive_path):
    """Make local_path a symlink into Drive.

    Idempotent and safe to call after every clone. The first time, local_path
    is either absent (data/splits and the DeepDTA source dir are gitignored)
    or a plain directory git just created (data/splits/.gitkeep is tracked so
    the empty directory exists) - either way it gets replaced by the symlink.
    On every run after that, local_path is already the symlink and this is a
    no-op, so re-running this cell after a reconnect never re-downloads or
    re-builds anything that is already sitting in Drive.
    """
    if os.path.islink(local_path):
        return
    if os.path.exists(local_path):
        !rm -rf {local_path}
    parent = os.path.dirname(local_path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    os.symlink(drive_path, local_path)


ensure_symlink('results', DRIVE_RESULTS)
ensure_symlink('data/splits', DRIVE_SPLITS)
ensure_symlink('src/data/baselines/deepdta/data', DRIVE_DEEPDTA_DATA)

print('results          ->', os.path.realpath('results'))
print('data/splits      ->', os.path.realpath('data/splits'))
print('deepdta data     ->', os.path.realpath('src/data/baselines/deepdta/data'))

## 4. Sanity-check the environment

566 tests in about 15 seconds. Cheap insurance before spending GPU hours on a broken
checkout.

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -q 2>&1 | tail -3

## 5. Fetch the DAVIS/KIBA source files

These are gitignored (they belong to DeepDTA, not to us), so they are downloaded from
the original repo. KIBA is fetched too because `build_splits` builds both datasets in
one pass - we only *train* on DAVIS.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'

for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
        # else: already in Drive from a previous session - nothing to fetch

!python -m src.data.load_data

Expected output, exactly:

```
davis: 30056 measured pairs, 68 unique drugs, 442 unique targets, Y range [5.000, 10.796]
kiba: 118254 measured pairs, 2111 unique drugs, 229 unique targets, ...
```

If the counts differ, stop - the downstream splits would be built on the wrong data.

## 6. Build the splits

Leakage checks run automatically and must all say OK. Cheap to rebuild every session
(same seed, deterministic) even though it now lives in Drive - a few seconds either way.

In [ ]:
!python -m src.data.build_splits
!python -m src.model.run_grid --preflight --task binary --datasets davis 2>&1 | tail -5

Expected for DAVIS (verified):

```
random       train=21039  valid=3006  test=6011
cold_drug    train=21658  valid=2652  test=5746
cold_target  train=21080  valid=2992  test=5984
cold_pair    train=15190  valid=264   test=1144
```

`cold_pair` validation really is only 264 rows - inherent to requiring both the drug and
the target to be unseen. Early stopping on it is noisy across seeds; that is a known
property, not a bug.

Preflight must end with `ready_to_launch: True`.

## 7. Train

Cheapest model first, so a shape error surfaces in minutes rather than hours. Each of the
three cells below is independent and **resumable** - a finished cell is skipped, so if
Colab disconnects, just re-run the notebook from the top and it picks up where it stopped.

Run them one at a time and check the output before moving on.

### 7a. DeepDTA - 12 cells, ~1-2 h

The accuracy anchor. No attention, so it never enters the explanation axis.

In [ ]:
!chmod +x run_davis_grid.sh
!./run_davis_grid.sh deepdta

**Check before continuing:** DAVIS random AUROC should land around 0.85-0.90, and
cold_pair clearly lower. Anything at 0.99 means a labelling or metric bug, not a good
model - stop and investigate rather than spending 20 more hours on it.

### 7b. ColdSite-DTI - 12 cells, ~3-7 h

Our own model. `run_grid` validates the first cell end to end before launching the other 11.

In [ ]:
# 64 needs 8.7 GB and fits a T4. On a smaller card use COLDSITE_BATCH=16.
!COLDSITE_BATCH=64 ./run_davis_grid.sh coldsite

### 7c. HyperAttentionDTI - 12 cells, ~7-19 h

The long one. Start it when you can leave the tab open.

In [ ]:
# batch 32 x accum 1 is the literal vendored configuration and needs ~5 GB.
# On a 4 GB card: HAT_BATCH=8 HAT_ACCUM=4 - same effective batch, same gradient.
!HAT_BATCH=32 HAT_ACCUM=1 ./run_davis_grid.sh hyperattentiondti

## 8. Analysis

Minutes, not hours. Run these once the grid above is complete.

### 8a. Faithfulness, then the ladder

In that order: faithfulness writes the accuracy JSON that the ladder refuses to draw a
figure without. Fidelity plotted without accuracy is half the paper's claim.

In [ ]:
for seed in (1, 2, 3):
    print(f'\n===== seed {seed} =====')
    !python -m src.evaluation.run_faithfulness --dataset davis --seed {seed} \
        --task binary --checkpoint-dir results
    !python -m src.evaluation.run_ladder --dataset davis --seed {seed} --task binary \
        --ground-truth data/davis_ground_truth_sites.json \
        --checkpoint-dir results \
        --accuracy-json results/accuracy_davis_seed{seed}.json

### 8b. The audit grid

The paper's central table. Holm-Bonferroni is applied once over the whole family.

In [ ]:
!python -m src.evaluation.run_audit \
    --models coldsite_dti,hyperattentiondti,uniform_control \
    --datasets davis --seeds 1,2,3 --task binary \
    --ground-truth data/davis_ground_truth_sites.json

### 8c. The confound control, reported both ways

The non-kinase arm is a **transfer** condition, not a stratification - DAVIS has zero
non-kinase targets to stratify into. Run it with and without the cotransport-ion
exclusion and report both; the difference is 29 of 396 panel positions.

In [ ]:
for model in ('coldsite_dti', 'hyperattentiondti'):
    for extra in ('', '--exclude-cotransport-ions'):
        print(f'\n===== {model} {extra or "(all ligands)"} =====')
        !python -m src.evaluation.run_control --model {model} \
            --dataset davis --seed 1 --task binary {extra}

## 9. What landed

In [ ]:
import glob

print('checkpoints :', len(glob.glob('results/*.pt')), '(expect 36)')
print('run results :', len(glob.glob('results/*_results.json')), '(expect 36)')
print()
for path in sorted(glob.glob('results/*.md')):
    print(' ', path)

audit = sorted(glob.glob('results/audit_davis*.md'))
if audit:
    print('\n' + '=' * 70)
    print(open(audit[0]).read())

## 10. Send the results back

Pushes the summary tables and figures-worth of JSON to the branch, so the analysis can
continue from the repo rather than from screenshots.

Needs a GitHub token in **Colab Secrets** (key icon, left sidebar) named `GITHUB_TOKEN`,
with `repo` scope. Skip this cell if you would rather download from Drive - everything is
already in your `coldsite-results` folder.

In [ ]:
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')

!git config user.email 'mahimagarwal5@gmail.com'
!git config user.name 'Mahim'

# results/*.md and *.csv are tracked. The summary JSONs are force-added on purpose:
# results/* is otherwise ignored so that checkpoints and DUMMY runs can never be
# committed. The 36 per-run JSONs stay out - they are regenerable and bulky.
!git add results/*.md results/*.csv 2>/dev/null
!git add -f results/audit_*.json results/ladder_*.json \
    results/faithfulness_*.json results/control_*.json results/accuracy_*.json 2>/dev/null
!git status --short | head -20

!git commit -m 'DAVIS binary grid: 36 runs, ladder, audit and control results' || echo 'nothing to commit'
!git push https://{token}@github.com/udayraj1238/ColdSite-DTI.git HEAD:{BRANCH}